# Augmentasi latent di embedding TabPFN-3.5 (2×T4)
Semua kode ada di notebook ini. Dataset Kaggle hanya berisi data (`label_01/`, `label_10/`, `label_30/`).

Syarat: dataset ditambahkan, Accelerator **GPU T4 x2**, Internet **on**. Bobot TabPFN-3.5 diunduh dari Hugging Face (`Prior-Labs/tabpfn_3_5`, lisensi non-komersial).

Cell `%%writefile` menulis kode ke `/kaggle/working/` supaya bisa dijalankan satu proses per GPU.

In [ ]:
!pip install -q -U tabpfn tabpfn-extensions
!pip show tabpfn | head -2
!nvidia-smi -L

In [ ]:
import os
os.environ['TABPFN_NO_BROWSER'] = '1'  # tidak ada browser di Kaggle
# Opsional: kalau paket tetap minta token, tambahkan secret TABPFN_TOKEN (API key Prior Labs)
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['TABPFN_TOKEN'] = UserSecretsClient().get_secret('TABPFN_TOKEN')
except Exception:
    print('tanpa TABPFN_TOKEN: bobot diambil dari Hugging Face')
# Lokasi data (folder yang berisi label_01/). Kosongkan untuk dicari otomatis di /kaggle/input.
os.environ['DATA_DIR'] = ''
!find /kaggle/input -maxdepth 6 -type d -name label_01

## Fungsi proyek (salinan persis)
Loader, metrik, dan operator latent yang sama dengan eksperimen 5.

In [ ]:
%%writefile /kaggle/working/common.py
"""Salinan persis fungsi proyek (git 2385029, working tree): experiments/usulan.py (load, scores),
experiments/rankup/tabcutmix.py (same_rank_donors), latent.py (operator latent). Jangan diubah di sini."""
from __future__ import annotations

import math

import numpy as np
import torch
import torch.nn.functional as F
from scipy.stats import spearmanr
from sklearn.metrics import (accuracy_score, cohen_kappa_score, f1_score, mean_absolute_error,
                             mean_squared_error, r2_score)


def load(root, name):
    with np.load(root / f'{name}.npz', allow_pickle=False) as split:
        return split['X'].astype(np.float32), (split['y'] if 'y' in split else None)


def scores(task, y, pred, levels=None):
    if task == 'classification':
        return {'macro_f1': float(f1_score(y, pred, average='macro', zero_division=0)),
                'accuracy': float(accuracy_score(y, pred))}
    result = {'mse': float(mean_squared_error(y, pred)), 'mae': float(mean_absolute_error(y, pred)),
              'r2': float(r2_score(y, pred)), 'spearman': float(spearmanr(y, pred).statistic)}
    if task == 'ordinal_regression':
        rounded = levels[np.abs(pred.reshape(-1, 1) - levels).argmin(axis=1)]
        result.update(qwk=float(cohen_kappa_score(y, rounded, labels=levels, weights='quadratic')),
                      ordinal_mae=float(mean_absolute_error(y, rounded)),
                      accuracy=float(accuracy_score(y, rounded)))
    return result


def same_rank_donors(pseudo_labels: torch.Tensor, confidence: torch.Tensor, cutoff: float):
    n = len(pseudo_labels)
    eligible = (pseudo_labels[:, None].eq(pseudo_labels[None, :])
                & confidence[None, :].ge(cutoff)
                & ~torch.eye(n, dtype=torch.bool, device=pseudo_labels.device))
    has_donor = eligible.any(dim=1)
    priorities = torch.rand((n, n), device=pseudo_labels.device)
    donors = priorities.masked_fill(~eligible, -1.0).argmax(dim=1)
    return torch.where(has_donor, donors, torch.arange(n, device=pseudo_labels.device)), has_donor


def rotate(h: torch.Tensor, u: torch.Tensor, c: torch.Tensor) -> torch.Tensor:
    """h' = ‖h‖ (c·v + √(1−c²)·û), v = h/‖h‖, û = bagian u yang tegak lurus v (dinormalkan). Norma tetap."""
    norm = h.norm(dim=-1, keepdim=True)
    v = h / norm.clamp_min(1e-12)
    u = u - (u * v).sum(-1, keepdim=True) * v
    u = u / u.norm(dim=-1, keepdim=True).clamp_min(1e-12)
    rotated = norm * (c * v + (1 - c ** 2).clamp_min(0).sqrt() * u)
    return torch.where(norm > 1e-6, rotated, h)


def angular(h: torch.Tensor, low: float, high: float) -> torch.Tensor:
    """Putar h ke arah acak sehingga cos(h, h') = c ~ U(low, high) tepat, norma tetap (ide pembimbing)."""
    u = torch.randn_like(h)  # urutan RNG sama dengan versi lama: hasil run angular lama tetap bisa diulang
    return rotate(h, u, torch.empty_like(h[..., :1]).uniform_(low, high))


def head_view(weight: torch.Tensor, rank: bool) -> torch.Tensor:
    """Bobot head (k, d, out) yang benar-benar dilihat loss. RankUp: output [regresi, ARC0, ARC1] dan
    peluang i > j hanya bergantung pada r_i − r_j, dengan r = h·a, a = W[..., 2] − W[..., 1]."""
    weight = weight.detach()
    return (weight[..., 2] - weight[..., 1])[..., None] if rank else weight


def head_direction(h: torch.Tensor, weight: torch.Tensor, rank: bool, adversarial: bool) -> torch.Tensor:
    """Arah di ruang h yang paling mengubah output head linear (closed form, tanpa backward tambahan).

    rank: skor r = h·a didorong ke median batch, sehingga urutan antar sampel paling sulit dibedakan.
    klasifikasi: dari bobot kelas teratas ke kelas kedua (batas keputusan terdekat).
    adversarial=False: tanda arah diacak (kontrol 'headrot': sadar-head tapi tidak adversarial).
    """
    w = head_view(weight, rank)                                       # (k, d, o)
    if rank:
        a = w[..., 0]                                                 # (k, d)
        r = (h * a).sum(-1)                                           # (B, k)
        sign = -torch.sign(r - r.median(dim=0).values)
        direction = a.expand_as(h)
    else:
        top = torch.einsum('bkd,kdo->bko', h, w).topk(2, dim=-1).indices
        wt, member = w.transpose(1, 2), torch.arange(w.shape[0], device=h.device)  # (k, o, d)
        direction = wt[member, top[..., 1]] - wt[member, top[..., 0]]  # (B, k, d)
        sign = torch.ones(h.shape[:2], device=h.device)
    if not adversarial:
        sign = torch.randint(0, 2, h.shape[:2], device=h.device) * 2.0 - 1
    sign = torch.where(sign == 0, torch.ones_like(sign), sign)        # sampel tepat di median
    return sign[..., None] * direction


def perturb(op: str, h: torch.Tensor, strength, pseudo_labels=None, confidence=None,
            donor_confidence: float = 0.8, head=None, rank: bool = True) -> torch.Tensor:
    """h: (batch, submodel, dim). strength: angka; (low, high) cosine untuk 'angular';
    (low, high) angular distance d ∈ [0, 1] (θ = π·d) untuk 'advrot'/'headrot', yang juga butuh head (k, d, out)."""
    chosen = torch.rand_like(h) < strength if op in ('zero_out', 'scarf', 'tabcutmix') else None
    if op == 'gaussian':  # skala noise ikut sebaran tiap unit di batch, jadi strength tanpa satuan
        return h + strength * h.std(dim=0, keepdim=True, unbiased=False) * torch.randn_like(h)
    if op == 'zero_out':
        return h.masked_fill(chosen, 0.0)
    if op == 'scarf':  # unit terpilih diambil dari baris acak lain di batch
        return torch.where(chosen, h.gather(0, torch.randint(len(h), h.shape, device=h.device)), h)
    if op == 'tabcutmix':  # unit terpilih diambil dari baris dengan pseudo-label sama
        donors, has_donor = same_rank_donors(pseudo_labels, confidence, donor_confidence)
        return torch.where(chosen & has_donor[:, None, None], h[donors], h)
    if op == 'dropout':  # UniMatch / CCT
        return F.dropout(h, p=strength, training=True)
    if op == 'fnoise':  # CCT F-Noise: z + z·U(−s, s)
        return h + h * (2 * torch.rand_like(h) - 1) * strength
    if op == 'angular':
        return angular(h, *strength)
    if op in ('advrot', 'headrot'):
        d = torch.empty_like(h[..., :1]).uniform_(*strength)
        return rotate(h, head_direction(h, head, rank, op == 'advrot'), torch.cos(math.pi * d))
    raise ValueError(f'operator latent tidak dikenal: {op!r}')


## Eksperimen

In [ ]:
%%writefile /kaggle/working/tabpfn_latent.py
"""Augmentasi latent di ruang embedding TabPFN-3.5, dengan TabPFN sebagai teacher untuk data unlabeled (2×T4).
Ditulis oleh notebook kaggle_tabpfn_latent.ipynb; ubah di notebook, bukan di file ini.

Per (dataset, label %, seed): TabPFN-3.5 di-fit pada data berlabel; embedding data berlabel diambil out-of-fold
(5 fold), embedding unlabeled/val/test dan prediksi teacher dari model penuh. Lalu head MLP dilatih di atas embedding:
loss berlabel + distilasi ke teacher di data unlabeled, dengan operator latent proyek di layer tersembunyi head.
Varian: tabpfn (teacher saja), head_sup, head_distill (tanpa augmentasi), head_<op>. Checkpoint dipilih dengan
validation, test hanya dilaporkan. Run yang sudah ada di hasil/*.jsonl dilewati.
"""
from __future__ import annotations

import argparse
import json
import os
import subprocess
import sys
import time
from pathlib import Path

import numpy as np

DATASETS, PERCENTS, SEEDS = ('nhanes', 'pendigits', 'superconductivity'), (1, 10, 30), (42, 123, 456)
METRIC = {'nhanes': ('qwk', 1), 'pendigits': ('macro_f1', 1), 'superconductivity': ('mse', -1)}  # (kunci, tanda)
# strength latent_strong dari GRID eksperimen latent (5-augmentasi-latent/kode/grid.py)
STRENGTH = {'gaussian': 0.5, 'zero_out': 0.5, 'scarf': 0.5, 'tabcutmix': 0.5, 'dropout': 0.5, 'fnoise': 0.3, 'angular': (0.3, 0.5)}
OUT = Path('/kaggle/working')
CHUNK = 2000  # baris per panggilan TabPFN, supaya 15k baris Superconductivity muat di T4


def find_data() -> Path:
    """Folder yang berisi label_01/, label_10/, label_30/ (DATA_DIR, atau dicari di /kaggle/input)."""
    if os.environ.get('DATA_DIR'):
        return Path(os.environ['DATA_DIR'])
    for path in sorted(Path('/kaggle/input').glob('**/label_01')):
        if (path / 'nhanes' / 'manifest.json').exists():
            return path.parent
    raise SystemExit('folder data (label_01/nhanes/manifest.json) tidak ditemukan di /kaggle/input')


DATA = find_data()


def variants() -> dict:
    """nama -> (operator, strength, bobot unlabeled)."""
    table = {'head_sup': (None, None, 0.0), 'head_distill': (None, None, 1.0)}
    table.update({f'head_{op}': (op, value, 1.0) for op, value in STRENGTH.items()})
    return table


# ---------------------------------------------------------------- TabPFN: teacher dan embedding

def read(dataset: str, percent: int):
    from common import load
    root = DATA / f'label_{percent:02d}' / dataset
    manifest = json.loads((root / 'manifest.json').read_text(encoding='utf-8'))
    task = manifest['task_type']
    ids = {str(c): i for i, c in enumerate(manifest.get('classes', []))}
    target = (lambda y: np.asarray([ids[v] for v in np.asarray(y).astype(str)], dtype=np.int64)) \
        if task == 'classification' else (lambda y: np.asarray(y, dtype=np.float32).reshape(-1))
    splits = {name: load(root, name) for name in ('train_labeled', 'train_unlabeled', 'validation', 'test')}
    splits = {name: (x, None if y is None else target(y)) for name, (x, y) in splits.items()}
    levels = np.asarray(manifest['target_values'], dtype=np.int64) if task == 'ordinal_regression' else None
    return task, splits, levels


WEIGHTS = ('Prior-Labs/tabpfn_3_5', 'tabpfn-v3.5-20260909.safetensors')  # satu checkpoint untuk klasifikasi dan regresi


def weights() -> str:
    """Bobot TabPFN-3.5 dari Hugging Face (repo tidak di-gate); file sudah ada di cache tidak diunduh lagi."""
    from huggingface_hub import hf_hub_download
    # local_dir: file asli bernama *.safetensors, bukan symlink ke blob cache HF tanpa ekstensi
    # (tabpfn menentukan format dari nama file; lewat symlink ia salah memakai torch.load)
    return hf_hub_download(*WEIGHTS, local_dir=OUT / 'weights')


def tabpfn(task: str, seed: int):
    from tabpfn import TabPFNClassifier, TabPFNRegressor
    cls = TabPFNClassifier if task == 'classification' else TabPFNRegressor
    return cls(model_path=weights(), device='cuda', random_state=seed)


def mean_members(e, n: int) -> np.ndarray:
    """(estimator, baris, dim) -> (baris, dim): rata-rata anggota ensemble TabPFN."""
    e = np.asarray(e)
    assert e.ndim == 3 and e.shape[1] == n, e.shape
    return e.mean(axis=0).astype(np.float32)


def chunked(fn, x: np.ndarray) -> np.ndarray:
    return np.concatenate([fn(x[s:s + CHUNK]) for s in range(0, len(x), CHUNK)])


def prepare(dataset: str, percent: int, seed: int) -> dict:
    """Embedding + target teacher, disimpan di cache supaya run ulang tidak menghitung TabPFN lagi.

    TabPFNEmbedding (tabpfn-extensions): fit_transform memberi embedding data berlabel out-of-fold (5 fold), jadi
    tidak ada baris yang di-embed oleh konteks yang memuat dirinya sendiri; transform memakai model penuh (model_),
    yang juga menjadi teacher.
    """
    cache = OUT / 'cache' / f'{dataset}_{percent:02d}_s{seed}.npz'
    if cache.exists():
        with np.load(cache) as saved:
            return dict(saved)
    from tabpfn_extensions.embedding import TabPFNEmbedding
    task, s, _ = read(dataset, percent)
    (x_lab, y_lab), (x_unl, _), (x_val, _), (x_test, _) = s.values()
    started = time.time()
    embedding = TabPFNEmbedding(n_fold=5, model=tabpfn(task, seed), shuffle=True, random_state=seed)
    out = {'e_lab': mean_members(embedding.fit_transform(x_lab, y_lab), len(x_lab))}
    model = embedding.model_  # model penuh, di-fit pada semua data berlabel
    predict = model.predict_proba if task == 'classification' else model.predict
    for name, x in (('unl', x_unl), ('val', x_val), ('test', x_test)):
        out[f'e_{name}'] = chunked(lambda b: mean_members(embedding.transform(b), len(b)), x)
        out[f't_{name}'] = chunked(predict, x).astype(np.float32)
    out['tabpfn_seconds'] = np.float32(time.time() - started)
    cache.parent.mkdir(parents=True, exist_ok=True)
    np.savez(cache, **out)
    return out


# ---------------------------------------------------------------- head + augmentasi latent

def train_head(task, splits, levels, data, op, strength, unlabeled_weight, seed, steps, dataset):
    import torch
    import torch.nn.functional as F
    from torch import nn
    from common import perturb, scores

    torch.manual_seed(seed)
    np.random.seed(seed)
    dev = torch.device('cuda')
    y_lab, y_val, y_test = splits['train_labeled'][1], splits['validation'][1], splits['test'][1]
    classification = task == 'classification'
    # standarisasi embedding dengan statistik semua data latih (tanpa label), target regresi dengan data berlabel
    pool = np.concatenate([data['e_lab'], data['e_unl']])
    mu, sd = pool.mean(0), pool.std(0) + 1e-6
    t = lambda a: torch.as_tensor((a - mu) / sd, dtype=torch.float32, device=dev)
    xl, xu, xv, xt = t(data['e_lab']), t(data['e_unl']), t(data['e_val']), t(data['e_test'])
    if classification:
        yl, tu = torch.as_tensor(y_lab, device=dev), torch.as_tensor(data['t_unl'], device=dev)
        conf, pseudo = tu.max(dim=1)
        n_out = tu.shape[1]
    else:
        y_mu, y_sd = float(y_lab.mean()), float(y_lab.std() + 1e-6)
        yl = torch.as_tensor((y_lab - y_mu) / y_sd, device=dev)
        tu = torch.as_tensor((data['t_unl'] - y_mu) / y_sd, device=dev)
        # pseudo-label untuk donor TabCutMix: level terdekat (ordinal) atau desil prediksi teacher (regresi)
        raw = data['t_unl']
        bins = (np.abs(raw[:, None] - levels).argmin(1) if levels is not None
                else np.searchsorted(np.quantile(raw, np.linspace(0.1, 0.9, 9)), raw))
        pseudo, conf = torch.as_tensor(bins, device=dev), torch.ones(len(raw), device=dev)
        n_out = 1

    body = nn.Sequential(nn.Linear(xl.shape[1], 256), nn.ReLU(), nn.Dropout(0.1), nn.Linear(256, 256), nn.ReLU())
    head = nn.Linear(256, n_out)
    model = nn.ModuleDict({'body': body, 'head': head}).to(dev)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

    def forward(x, idx=None):
        h = model['body'](x)
        if op and idx is not None:  # (batch, 1, dim): bentuk yang dipakai latent.perturb
            h = perturb(op, h[:, None], strength, pseudo[idx], conf[idx])[:, 0]
        out = model['head'](h)
        return out if classification else out[:, 0]

    def predict(x):
        model.eval()
        with torch.no_grad():
            out = forward(x)
        model.train()
        return out.argmax(1).cpu().numpy() if classification else out.cpu().numpy() * y_sd + y_mu

    key, sign = METRIC[dataset]
    best, best_state, best_step, bad = -np.inf, None, 0, 0
    for step in range(1, steps + 1):
        il = torch.randint(len(xl), (min(32, len(xl)),), device=dev)
        out = forward(xl[il])
        loss = F.cross_entropy(out, yl[il]) if classification else F.mse_loss(out, yl[il])
        if unlabeled_weight:
            iu = torch.randint(len(xu), (224,), device=dev)  # uratio 7 seperti RankUp
            ou = forward(xu[iu], iu)
            unsup = (-(tu[iu] * F.log_softmax(ou, 1)).sum(1).mean() if classification
                     else F.mse_loss(ou, tu[iu]))
            loss = loss + unlabeled_weight * unsup
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()
        if step % 50 == 0:
            score = sign * scores(task, y_val, predict(xv), levels)[key]
            if score > best:
                best, best_step, bad = score, step, 0
                best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
            else:
                bad += 1
                if bad >= 20:  # 1000 langkah tanpa perbaikan di validation
                    break
    model.load_state_dict(best_state)
    return {'validation': scores(task, y_val, predict(xv), levels), 'test': scores(task, y_test, predict(xt), levels),
            'best_step': best_step}


# ---------------------------------------------------------------- loop per GPU

def worker(groups, out_path: Path, steps: int):
    from importlib.metadata import version as package_version
    from common import scores
    version = package_version('tabpfn')
    table = variants()
    names = ['tabpfn', *table]
    done = set()
    if out_path.exists():
        done = {(r['dataset'], r['label_percent'], r['seed'], r['name'])
                for r in map(json.loads, out_path.read_text(encoding='utf-8').splitlines())}
    out_path.parent.mkdir(parents=True, exist_ok=True)
    with out_path.open('a', encoding='utf-8') as sink:
        for dataset, percent, seed in groups:
            todo = [n for n in names if (dataset, percent, seed, n) not in done]
            if not todo:
                continue
            task, splits, levels = read(dataset, percent)
            data = prepare(dataset, percent, seed)
            for name in todo:
                started = time.time()
                if name == 'tabpfn':
                    pred = (lambda t: t.argmax(1)) if task == 'classification' else (lambda t: t)
                    result = {'validation': scores(task, splits['validation'][1], pred(data['t_val']), levels),
                              'test': scores(task, splits['test'][1], pred(data['t_test']), levels)}
                else:
                    op, strength, weight = table[name]
                    result = train_head(task, splits, levels, data, op, strength, weight, seed, steps, dataset)
                row = {'name': name, 'dataset': dataset, 'label_percent': percent, 'seed': seed, 'task': task,
                       **result, 'seconds': time.time() - started,
                       'tabpfn_seconds': float(data['tabpfn_seconds']), 'tabpfn_version': version, 'weights': WEIGHTS[1]}
                sink.write(json.dumps(row) + '\n')
                sink.flush()
                key = METRIC[dataset][0]
                print(f"[{os.environ.get('CUDA_VISIBLE_DEVICES', '-')}] {dataset} {percent}% s{seed} {name:14s} "
                      f"val {result['validation'][key]:.4f} test {result['test'][key]:.4f}", flush=True)


def summary(paths):
    import pandas as pd
    rows = [json.loads(line) for p in paths if p.exists() for line in p.read_text(encoding='utf-8').splitlines()]
    if not rows:
        print('belum ada hasil')
        return
    df = pd.DataFrame([{'name': r['name'], 'cell': f"{r['dataset']} {r['label_percent']}%",
                        'test': r['test'][METRIC[r['dataset']][0]]} for r in rows])
    table = df.groupby(['name', 'cell']).test.agg(lambda s: f'{s.mean():.3f}±{s.std():.3f}').unstack()
    print(table.to_string())


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--gpus', type=int, default=2)
    parser.add_argument('--worker', type=int, default=None)
    parser.add_argument('--steps', type=int, default=3000)
    parser.add_argument('--smoke', action='store_true', help='2 grup, 200 langkah, 1 GPU: cek sebelum run penuh')
    args = parser.parse_args()
    if args.smoke:
        path = OUT / 'hasil' / 'smoke.jsonl'
        path.unlink(missing_ok=True)
        worker([('nhanes', 10, 42), ('pendigits', 1, 42)], path, 200)
        summary([path])
        return
    weights()  # unduh sekali di proses induk, supaya dua worker tidak mengunduh 876 MB bersamaan
    groups = [(d, p, s) for p in PERCENTS for d in DATASETS for s in SEEDS]
    paths = [OUT / 'hasil' / f'results_gpu{i}.jsonl' for i in range(args.gpus)]
    if args.worker is not None:
        worker(groups[args.worker::args.gpus], paths[args.worker], args.steps)
        return
    # satu proses per GPU; tiap proses hanya melihat GPU-nya sendiri
    procs = [subprocess.Popen([sys.executable, __file__, '--gpus', str(args.gpus), '--worker', str(i),
                               '--steps', str(args.steps)], env={**os.environ, 'CUDA_VISIBLE_DEVICES': str(i)})
             for i in range(args.gpus)]
    codes = [p.wait() for p in procs]
    summary(paths)
    if any(codes):
        raise SystemExit(f'ada worker gagal: exit code {codes}')


if __name__ == '__main__':
    main()


## 1. Smoke test
2 grup, 200 langkah, 1 GPU. Kalau gagal di sini, kirim error-nya dulu.

In [ ]:
%%bash
# %%bash menghentikan Run All kalau smoke test gagal (!python tidak)
python /kaggle/working/tabpfn_latent.py --smoke

## 2. Run penuh
27 grup (3 dataset × 3 label % × 3 seed) × 10 varian, satu proses per GPU.

In [ ]:
!python /kaggle/working/tabpfn_latent.py

## 3. Unduh hasil
`/kaggle/working` kosong lagi di sesi baru, jadi simpan zip ini (atau Save Version).

In [ ]:
!cd /kaggle/working && zip -qr hasil_tabpfn_latent.zip hasil cache && ls -lh hasil_tabpfn_latent.zip